# Practical use for expedition data

In [ ]:
!pip -q install silero-vad
!pip -q install pympi-ling
!pip -q install transformers accelerate huggingface_hub safetensors pympi-ling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.7 MB/s eta 0:00:00


## Cutting long file to segments

In [ ]:
from pathlib import Path
import re
import unicodedata

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch

from silero_vad import load_silero_vad, get_speech_timestamps

In [ ]:
# metadata = pd.read_csv('metadata.tsv', sep='\t', encoding='utf-8-sig')
# metadata[metadata.resource == 'chuklang'].duration.describe()

In [ ]:
def load_audio_16k_mono(input_path, sr):
    input_path = Path(input_path)

    y, _ = librosa.load(
        str(input_path),
        sr=sr,
        mono=True
    )

    y = np.asarray(y, dtype=np.float32)
    y = np.nan_to_num(y)

    return y, sr


def add_non_overlapping_padding(segments, total_samples, sr, pad_sec):
    if len(segments) == 0:
        return []

    pad_samples = int(pad_sec * sr)

    padded_segments = []

    for i, (start, end) in enumerate(segments):
        if i == 0:
            left_limit = 0
        else:
            prev_end = segments[i - 1][1]
            left_limit = (prev_end + start) // 2

        if i == len(segments) - 1:
            right_limit = total_samples
        else:
            next_start = segments[i + 1][0]
            right_limit = (end + next_start) // 2

        padded_start = max(0, start - pad_samples, left_limit)
        padded_end = min(total_samples, end + pad_samples, right_limit)

        padded_segments.append({
            'core_start_sample': start,
            'core_end_sample': end,
            'segment_start_sample': padded_start,
            'segment_end_sample': padded_end,
            'core_start_sec': start / sr,
            'core_end_sec': end / sr,
            'segment_start_sec': padded_start / sr,
            'segment_end_sec': padded_end / sr,
            'core_duration': (end - start) / sr,
            'segment_duration': (padded_end - padded_start) / sr
        })

    return padded_segments


def detect_speech_intervals_silero(
    y,
    sr,
    threshold,
    min_speech_duration_ms,
    min_silence_duration_ms,
    speech_pad_ms=0
):
    model = load_silero_vad()

    speech_timestamps = get_speech_timestamps(
        torch.from_numpy(y),
        model,
        sampling_rate=sr,
        threshold=threshold,
        min_speech_duration_ms=min_speech_duration_ms,
        min_silence_duration_ms=min_silence_duration_ms,
        speech_pad_ms=speech_pad_ms,
        return_seconds=False
    )

    intervals = [
        (item['start'], item['end'])
        for item in speech_timestamps
    ]

    return intervals


def segment_long_audio_for_eaf_silero(
    input_path,
    output_dir,
    sr=16000,
    threshold=0.35,
    min_speech_duration_ms=500,
    min_silence_duration_ms=500,
    pad_sec=0.2,
    save_converted_audio=True
):
    input_path = Path(input_path)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    recording_id = input_path.stem

    y, sr = load_audio_16k_mono(
        input_path,
        sr=sr
    )

    converted_path = None

    if save_converted_audio:
        converted_path = output_dir / f'{recording_id}.wav'
        sf.write(converted_path, y, sr)

    core_segments = detect_speech_intervals_silero(
        y,
        sr=sr,
        threshold=threshold,
        min_speech_duration_ms=min_speech_duration_ms,
        min_silence_duration_ms=min_silence_duration_ms,
        speech_pad_ms=0
    )

    segment_rows = add_non_overlapping_padding(
        core_segments,
        total_samples=len(y),
        sr=sr,
        pad_sec=pad_sec
    )

    rows = []

    for i, row in enumerate(segment_rows, start=1):
        rows.append({
            'recording_id': recording_id,
            'segment_id': i,

            'segment_start_sec': round(row['segment_start_sec'], 3),
            'segment_end_sec': round(row['segment_end_sec'], 3),
            'segment_duration': round(row['segment_duration'], 3),

            'core_start_sec': round(row['core_start_sec'], 3),
            'core_end_sec': round(row['core_end_sec'], 3),
            'core_duration': round(row['core_duration'], 3),

            'segment_start_sample': row['segment_start_sample'],
            'segment_end_sample': row['segment_end_sample'],
            'core_start_sample': row['core_start_sample'],
            'core_end_sample': row['core_end_sample'],

            'asr_text': ''
        })

    segments_df = pd.DataFrame(rows)

    manifest_path = output_dir / f'{recording_id}_segments_for_eaf.csv'

    segments_df.to_csv(
        manifest_path,
        index=False,
        encoding='utf-8-sig'
    )

    print(f'Input audio duration: {len(y) / sr:.2f} sec')
    print(f'Created segments: {len(segments_df)}')
    print(f'Manifest: {manifest_path}')

    if converted_path is not None:
        print(f'Converted audio: {converted_path}')

    if len(segments_df) > 0:
        display(segments_df['segment_duration'].describe())

    return segments_df, y, sr, converted_path

In [ ]:
input_path = 'Water cart.wav'

segments_df, y, sr, converted_path = segment_long_audio_for_eaf_silero(
    input_path=input_path,
    output_dir='segmented_chuklang',
    save_converted_audio=True
)

Input audio duration: 193.55 sec
Created segments: 37
Manifest: segmented_chuklang/Water cart_segments_for_eaf.csv
Converted audio: segmented_chuklang/Water cart.wav


,segment_duration
count,37.000000
mean,4.572865
std,4.136459
min,0.912000
25%,1.680000
50%,2.992000
75%,5.904000
max,17.808000


## Get eaf file with empty annotations

In [ ]:
from pathlib import Path

import pandas as pd
import pympi

In [ ]:
def create_eaf_from_segments(segments_df, media_path, output_eaf_path,
    tier_id='segments', use_core_boundaries=False, label_segments=False):
    media_path = Path(media_path)
    output_eaf_path = Path(output_eaf_path)

    eaf = pympi.Elan.Eaf()

    eaf.add_linked_file(str(media_path))
    eaf.add_tier(tier_id)

    if use_core_boundaries:
        start_col = 'core_start_sec'
        end_col = 'core_end_sec'
    else:
        start_col = 'segment_start_sec'
        end_col = 'segment_end_sec'

    for _, row in segments_df.iterrows():
        start_ms = int(round(row[start_col] * 1000))
        end_ms = int(round(row[end_col] * 1000))

        if end_ms <= start_ms:
            continue

        if label_segments:
            segment_id = int(row['segment_id'])
            duration = row['segment_duration']
            value = f'seg_{segment_id:04d} [{duration:.2f}s]'
        else:
            value = ''

        eaf.add_annotation(tier_id, start_ms, end_ms, value)

    eaf.to_file(str(output_eaf_path))

    print(f'Saved EAF: {output_eaf_path}')

In [ ]:
recording_id = str(segments_df['recording_id'].iloc[0])

create_eaf_from_segments(
    segments_df=segments_df,
    media_path=converted_path,
    output_eaf_path=f'segmented_chuklang/{recording_id}.eaf'
)

Saved EAF: segmented_chuklang/Water cart.eaf


## ASR inference

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pympi

from huggingface_hub import snapshot_download
from transformers import AutoProcessor, Wav2Vec2ForCTC

In [ ]:
HF_MODEL_REPO_ID = 'tadgeis/mms-1b-all-ckt-best-cer-model'
TARGET_LANG = 'ckt'

In [ ]:
def download_model_locally(repo_id, local_dir, token=None):
    local_dir = snapshot_download(
        repo_id=repo_id,
        repo_type='model',
        local_dir=local_dir,
        token=token,
        ignore_patterns=[
            'last-checkpoint/*',
            'optimizer.pt',
            'scheduler.pt',
            'rng_state.pth',
            'scaler.pt',
            'trainer_state.json'
        ]
    )

    print(f'Downloaded to: {local_dir}')

    return local_dir


def load_mms_asr_model(
    model_source,
    target_lang='ckt',
    device=None,
    use_fp16=True,
    token=None
):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    torch_dtype = torch.float16 if device == 'cuda' and use_fp16 else torch.float32

    processor = AutoProcessor.from_pretrained(
        model_source,
        target_lang=target_lang,
        token=token
    )

    model = Wav2Vec2ForCTC.from_pretrained(
        model_source,
        target_lang=target_lang,
        ignore_mismatched_sizes=True,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        token=token
    )

    processor.tokenizer.set_target_lang(target_lang)

    try:
        model.load_adapter(target_lang)
    except Exception as error:
        print(f'Adapter loading warning: {error}')
        print('Если from_pretrained уже загрузил нужный adapter, это может быть не критично.')

    model.to(device)
    model.eval()

    print(f'Model source: {model_source}')
    print(f'Target language: {target_lang}')
    print(f'Device: {device}')
    print(f'Model dtype: {next(model.parameters()).dtype}')

    return processor, model, device

### Use from HuggingFace

In [ ]:
processor, model, device = load_mms_asr_model(model_source=HF_MODEL_REPO_ID, target_lang=TARGET_LANG, use_fp16=True)

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.ckt.safetensors:   0%|          | 0.00/8.85M [00:00<?, ?B/s]

Model source: tadgeis/mms-1b-all-ckt-best-cer-model
Target language: ckt
Device: cpu
Model dtype: torch.float32


### Download and use locally

In [ ]:
local_model_dir = download_model_locally(repo_id=HF_MODEL_REPO_ID, local_dir='/content/models/mms-1b-all-ckt-best-cer-model')

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded to: /content/models/mms-1b-all-ckt-best-cer-model


In [ ]:
processor, model, device = load_mms_asr_model(model_source=local_model_dir, target_lang=TARGET_LANG, use_fp16=True)

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Model source: /content/models/mms-1b-all-ckt-best-cer-model
Target language: ckt
Device: cpu
Model dtype: torch.float32


### ASR implementation itself

In [ ]:
def transcribe_segments(segments_df, y, sr, processor, model, device, batch_size=2,
    start_sample_col='segment_start_sample', end_sample_col='segment_end_sample',
    text_col='asr_text', sort_by_duration=True):
    segments_df = segments_df.copy()

    if text_col not in segments_df.columns:
        segments_df[text_col] = ''

    work_df = segments_df.reset_index().rename(columns={'index': '_original_index'})

    if sort_by_duration and 'segment_duration' in work_df.columns:
        work_df = work_df.sort_values('segment_duration')

    predictions = {}

    model_dtype = next(model.parameters()).dtype
    use_cuda_autocast = device == 'cuda' and model_dtype == torch.float16

    for batch_start in range(0, len(work_df), batch_size):
        batch_df = work_df.iloc[batch_start:batch_start + batch_size]

        audio_batch = []

        for _, row in batch_df.iterrows():
            start_sample = int(row[start_sample_col])
            end_sample = int(row[end_sample_col])

            audio = y[start_sample:end_sample]
            audio = np.asarray(audio, dtype=np.float32)

            audio_batch.append(audio)

        inputs = processor(
            audio_batch,
            sampling_rate=sr,
            padding=True,
            return_tensors='pt'
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            if use_cuda_autocast:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(**inputs).logits
            else:
                logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        batch_texts = processor.batch_decode(predicted_ids)

        for original_index, text in zip(batch_df['_original_index'], batch_texts):
            predictions[int(original_index)] = text

        print(f'Transcribed {min(batch_start + batch_size, len(work_df))}/{len(work_df)} segments')

    for index, text in predictions.items():
        segments_df.loc[index, text_col] = text

    return segments_df


def save_asr_segments_csv(segments_df_asr, output_dir, recording_id=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if recording_id is None:
        recording_id = str(segments_df_asr['recording_id'].iloc[0])

    output_csv_path = output_dir / f'{recording_id}_segments_with_asr.csv'

    segments_df_asr.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    print(f'Saved CSV: {output_csv_path}')

    return output_csv_path

In [ ]:
name = 'Water cart'
segments_df = pd.read_csv(f'segmented_chuklang/{name}_segments_for_eaf.csv')
y, sr = load_audio_16k_mono(f'segmented_chuklang/{name}.wav', sr=16000)

segments_df_asr = transcribe_segments(segments_df=segments_df, y=y, sr=sr,
    processor=processor, model=model, device=device, batch_size=2)

Transcribed 2/37 segments
Transcribed 4/37 segments
Transcribed 6/37 segments
Transcribed 8/37 segments
Transcribed 10/37 segments
Transcribed 12/37 segments
Transcribed 14/37 segments
Transcribed 16/37 segments
Transcribed 18/37 segments
Transcribed 20/37 segments
Transcribed 22/37 segments
Transcribed 24/37 segments
Transcribed 26/37 segments
Transcribed 28/37 segments
Transcribed 30/37 segments
Transcribed 32/37 segments
Transcribed 34/37 segments
Transcribed 36/37 segments
Transcribed 37/37 segments


/tmp/ipykernel_6811/721242200.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'э' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  segments_df.loc[index, text_col] = text


### Replacing cyrillic to IPA

In [ ]:
VOWEL_PAIRS_RULES = {
    "э'": 'ʔe',
    "а'": 'ʔa',
    "о'": 'ʔo',
    "и'": 'ʔi',
    "ы'": 'ʔə',
    "у'": 'ʔu',
    'ъя': 'ja',
    'ъе': 'je',
    'ъё': 'jo',
    'ъю': 'ju',
}


IOTATED_VOWEL_RULES = {
    'я': ('a', 'ja'),
    'е': ('e', 'je'),
    'ё': ('o', 'jo'),
    'ю': ('u', 'ju'),
}


CYRILLIC_VOWELS = {
    'а', 'о', 'э', 'е', 'ё', 'и', 'ы', 'у', 'ю', 'я',
}


CYR_TO_CHUKLANG_CHAR_RULES = {
    'ъ': 'ʔ',
    'ь': 'ʔ',

    'ӈ': 'ŋ',
    'ӄ': 'q',
    'ԓ': 'ɬ',
    'Ԓ': 'ɬ',

    'а': 'a',
    'о': 'o',
    'э': 'e',
    'и': 'i',
    'ы': 'ə',
    'у': 'u',

    'т': 't',
    'д': 'd',
    'л': 'ɬ',
    'м': 'm',
    'н': 'n',
    'в': 'w',
    'к': 'k',
    'г': 'ɣ',
    'ч': 's',
    'с': 's',
    'р': 'r',
    'б': 'b',
    'п': 'p',
    'й': 'j',
    'ж': 'ž',
    'з': 'z',
    'ф': 'f',
    'х': 'x',
    'ш': 'š',
}

In [ ]:
def remove_punctuation_keep_apostrophe(text):
    chars = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith('P') and char != "'":
            chars.append(' ')
        else:
            chars.append(char)

    return ''.join(chars)


def normalize_text_basic(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = text.replace('\u00a0', ' ')
    text = remove_punctuation_keep_apostrophe(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def transliterate_word_cyrillic_to_chuklang(word):
    result = []
    i = 0

    while i < len(word):
        two_chars = word[i:i + 2]

        if two_chars in VOWEL_PAIRS_RULES:
            result.append(VOWEL_PAIRS_RULES[two_chars])
            i += 2
            continue

        char = word[i]

        if char in IOTATED_VOWEL_RULES:
            plain_value, iotated_value = IOTATED_VOWEL_RULES[char]

            is_word_initial = i == 0
            is_after_vowel = i > 0 and word[i - 1] in CYRILLIC_VOWELS

            if is_word_initial or is_after_vowel:
                result.append(iotated_value)
            else:
                result.append(plain_value)

            i += 1
            continue

        result.append(CYR_TO_CHUKLANG_CHAR_RULES.get(char, char))
        i += 1

    return ''.join(result)


def transliterate_cyrillic_to_chuklang(text):
    text = normalize_text_basic(text)

    words = text.split()

    words = [
        transliterate_word_cyrillic_to_chuklang(word)
        for word in words
    ]

    return ' '.join(words)

In [ ]:
segments_df_asr['asr_text_ipa'] = segments_df_asr['asr_text'].apply(transliterate_cyrillic_to_chuklang)

segments_df_asr[
    [
        'recording_id',
        'segment_start_sec',
        'segment_end_sec',
        'asr_text',
        'asr_text_ipa'
    ]
].head()

,recording_id,segment_start_sec,segment_end_sec,asr_text,asr_text_ipa
0,Water cart,0.000,12.744,вашгын сочасныатов мыновароскажыса потомкаконв...,wašɣən sosasnəatow mənowaroskažəsa potomkakonw...
1,Water cart,13.912,26.216,люутэ вулӄытвика ӈэръамытлыӈычьэты галякэ очоч...,ɬuute wuɬqətwika ŋerʔamətɬəŋəsʔetə ɣaɬake osos...
2,Water cart,27.768,28.776,нэ,ne
3,Water cart,28.984,30.088,ыръэрэ,ərʔere
4,Water cart,30.584,32.936,ԓьэленру кытэкэмӄовыԓьэтык,ɬʔeɬenru kətekemqowəɬʔetək


In [ ]:
recording_id = str(segments_df_asr['recording_id'].iloc[0])

asr_csv_path = Path('asr_results') / f'{recording_id}_asr_with_ipa.csv'
asr_csv_path.parent.mkdir(parents=True, exist_ok=True)

segments_df_asr.to_csv(asr_csv_path, index=False)

print(f'Saved CSV: {asr_csv_path}')

Saved CSV: asr_results/Water cart_asr_with_ipa.csv


## Get eaf file with annotations on chukchi

In [ ]:
def create_eaf_from_asr_segments(segments_df, media_path, output_eaf_path,
    tier_id='ASR_raw', text_col='asr_text', use_core_boundaries=False):
    media_path = Path(media_path)
    output_eaf_path = Path(output_eaf_path)

    output_eaf_path.parent.mkdir(parents=True, exist_ok=True)

    eaf = pympi.Elan.Eaf()

    eaf.add_linked_file(str(media_path))
    eaf.add_tier(tier_id)

    if use_core_boundaries:
        start_col = 'core_start_sec'
        end_col = 'core_end_sec'
    else:
        start_col = 'segment_start_sec'
        end_col = 'segment_end_sec'

    for _, row in segments_df.iterrows():
        start_ms = int(round(float(row[start_col]) * 1000))
        end_ms = int(round(float(row[end_col]) * 1000))

        if end_ms <= start_ms:
            continue

        text = row.get(text_col, '')

        if pd.isna(text):
            text = ''

        eaf.add_annotation(tier_id, start_ms, end_ms, str(text))

    eaf.to_file(str(output_eaf_path))

    print(f'Saved EAF: {output_eaf_path}')

    return output_eaf_path


def create_eaf_from_asr_segments_with_two_tiers(
    segments_df,
    media_path,
    output_eaf_path,
    cyrillic_tier_id='ASR_cyrillic',
    ipa_tier_id='ASR_IPA',
    cyrillic_text_col='asr_text',
    ipa_text_col='asr_text_ipa',
    use_core_boundaries=False
):
    media_path = Path(media_path)
    output_eaf_path = Path(output_eaf_path)

    output_eaf_path.parent.mkdir(parents=True, exist_ok=True)

    eaf = pympi.Elan.Eaf()

    eaf.add_linked_file(str(media_path))
    eaf.add_tier(cyrillic_tier_id)
    eaf.add_tier(ipa_tier_id)

    if use_core_boundaries:
        start_col = 'core_start_sec'
        end_col = 'core_end_sec'
    else:
        start_col = 'segment_start_sec'
        end_col = 'segment_end_sec'

    for _, row in segments_df.iterrows():
        start_ms = int(round(float(row[start_col]) * 1000))
        end_ms = int(round(float(row[end_col]) * 1000))

        if end_ms <= start_ms:
            continue

        cyrillic_text = row.get(cyrillic_text_col, '')
        ipa_text = row.get(ipa_text_col, '')

        if pd.isna(cyrillic_text):
            cyrillic_text = ''

        if pd.isna(ipa_text):
            ipa_text = ''

        eaf.add_annotation(cyrillic_tier_id, start_ms, end_ms, str(cyrillic_text))
        eaf.add_annotation(ipa_tier_id, start_ms, end_ms, str(ipa_text))

    eaf.to_file(str(output_eaf_path))

    print(f'Saved EAF: {output_eaf_path}')

    return output_eaf_path

In [ ]:
recording_id = str(segments_df_asr['recording_id'].iloc[0])
media_path = f'segmented_chuklang/{recording_id}.wav'

create_eaf_from_asr_segments_with_two_tiers(
    segments_df=segments_df_asr,
    media_path=media_path,
    output_eaf_path=f'asr_results/{recording_id}.eaf',
    cyrillic_tier_id='ASR_cyrillic',
    ipa_tier_id='ASR_IPA',
    cyrillic_text_col='asr_text',
    ipa_text_col='asr_text_ipa'
)

Saved EAF: asr_results/Water cart.eaf


PosixPath('asr_results/Water cart.eaf')